# 00 Process Raw Data Full

- This process consists of aggregating all separeted files manually extracted from the data source
- After the first run, it will no longer be necessary for the rest of the pipeline

## Imports

In [18]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f

import os

import pandas as pd

from dotenv import load_dotenv

load_dotenv()

True

## Init pySpark

In [19]:
try:
    spark = SparkSession.builder.appName("raw_pipeline").getOrCreate()
except Exception as e:
    print(e)

## Aggregate all files

In [ ]:
raw_folder_path = r"../../data/raw/raw_separated"

In [21]:
for index, file_name in enumerate(os.listdir(raw_folder_path)):
    final_path = os.path.join(raw_folder_path, file_name)
    match_data = spark.read.format("csv").option("header", "true").load(final_path)
    if index == 0:
        df = match_data 
    else:
        df = df.unionByName(match_data, allowMissingColumns=True)

## Save dataframe

### Local

In [22]:
df.toPandas().to_csv(
    r"../../data/raw/tb_atp_matches.csv",
    index=False,
    sep=";",
    encoding="utf-8"
)

### Supabase

In [23]:
(
    df.write
    .format("jdbc")
    .option("url", os.getenv("JDBC_URL"))
    .option("dbtable", "raw.tb_atp_matches")
    .option("user", os.getenv("DB_USER"))
    .option("password", os.getenv("DB_PASSWORD"))
    .option("driver", "org.postgresql.Driver")
    .mode("append")
    .save()
)